# Scenario 3 Live — Hot Binary (ẑ) + Cold Orbiter (xy), 3D Makie

Live streaming animation of the Scenario 3 initial condition from
[`exhaust_nonzero_radial_velocity_ics.ipynb`](exhaust_nonzero_radial_velocity_ics.ipynb#Scenario-3).
The dashboard has one big 3D trajectory panel (mouse-rotatable `Axis3`),
a 2D phase-space sidebar with a dropdown selector, and a live energy-error
readout.

## Charges and coupling

| | $q_1$ | $q_2$ | $q_3$ |
|---|---|---|---|
| Charge | +1 | −2 | +1 |

- $k_{12} = q_1 q_2 = -2$ → attractive bound inner binary along $\hat z$.
- Binary net charge $q_1 + q_2 = -1$; orbiter $q_3 = +1$, so outer coupling
  $k_\mathrm{out} = (q_1+q_2)q_3 = -1$ → attractive outer orbit.

## Dynamics you should see

- Binary particles 1, 2 tracing short oscillatory segments near the origin,
  roughly along the $\hat z$-axis with small $xy$ wobble (from $L_B > 0$).
- Orbiter particle 3 sweeping a near-circle in the $xy$-plane at radius
  $\sim R_\mathrm{out}$, with the binary's COM counter-sweeping on the
  opposite side to keep total momentum zero.
- Live `ΔE/E₀ (max)` settling at the $10^{-4}$ %–$10^{-2}$ % level over a
  few outer periods, depending on `dt`.

## Requirements

A windowed Makie backend must be loaded. This notebook imports `GLMakie`;
replace with `WGLMakie` or `CairoMakie` if preferred. Close the GLMakie
window to stop the simulation.

See [`theory/NonZeroRadialVelocityBoundICs.md`](../../../theory/NonZeroRadialVelocityBoundICs.md)
§3 for the physics derivation.

In [1]:
using WeberElectrodynamics
using LinearAlgebra
using Printf
using GLMakie

[ Info: Precompiling WeberElectrodynamicsMakieExt [d22e3b8d-9b72-56d3-9429-0b8aa1f06b33](cache misses: include_dependency fsize change (1), incompatible header (6), mismatched flags (1), wrong julia version (2))
Precompiling packages...
   6436.0 ms  ✓ WeberElectrodynamics → WeberElectrodynamicsMakieExt
  1 dependency successfully precompiled in 6 seconds. 2 already precompiled.


## 1. Inline helpers (forward map, verification)

These are copied verbatim from the parent notebook so this file stands
alone.

In [2]:
_kappa(k::Real, i, j) = k
function _kappa(d::Dict, i, j)
    key = (min(i, j), max(i, j))
    return get(d, key, 1.0)
end

"""
    radial_velocity(E, L, r, k, mu, c, s)

Weber radial energy equation (★) from §2.1:  ṙ² = (E − k/r − L²/(2μr²)) / (μ − k/(2c²r)).
`s ∈ {+1, −1}` selects outbound / inbound.
"""
function radial_velocity(E::Real, L::Real, r::Real, k::Real, mu::Real, c::Real, s::Integer)
    @assert s == 1 || s == -1 "s must be ±1"
    num = E - k/r - L^2 / (2*mu*r^2)
    den = mu - k / (2*c^2 * r)
    den > 0 || throw(DomainError(r, "denominator μ − k/(2c²r) ≤ 0"))
    num >= 0 || throw(DomainError(r, "numerator < 0; r outside [r_p, r_a]"))
    return s * sqrt(num / den)
end

"""
    kepler_apsides(E, L, k, mu) -> (r_p, r_a)

Periapsis and apoapsis from the Kepler quadratic (L²/(2μ)) u² + k u − E = 0.
"""
function kepler_apsides(E::Real, L::Real, k::Real, mu::Real)
    @assert E < 0 "Need E < 0 (bound orbit)"
    @assert L != 0 "L = 0 reduces to a single fall radius"
    a_q = L^2 / (2*mu); b_q = k; c_q = -E
    disc = b_q^2 - 4*a_q*c_q
    @assert disc >= 0 "L too large for given E"
    u_plus  = (-b_q + sqrt(disc)) / (2*a_q)
    u_minus = (-b_q - sqrt(disc)) / (2*a_q)
    return (1/max(u_plus, u_minus), 1/min(u_plus, u_minus))
end

kepler_period(a::Real, mu::Real, k::Real) = 2π * sqrt(mu * a^3 / abs(k))

kepler_period (generic function with 1 method)

In [3]:
"""
    canonical_momentum(positions, velocities, masses, charges, kappas, c)

Forward map (F):  p_i = m_i v_i − Σ_{j≠i}(κ_ij q_i q_j / c²)(ṙ_ij/r_ij²)(r_i − r_j).
"""
function canonical_momentum(positions, velocities, masses, charges, kappas, c)
    n = length(positions)
    p = [collect(masses[i] .* velocities[i]) for i in 1:n]
    for i in 1:n, j in 1:n
        i == j && continue
        Δr = positions[i] .- positions[j]
        r  = norm(Δr)
        Δv = velocities[i] .- velocities[j]
        ṙ  = dot(Δr, Δv) / r
        kij = _kappa(kappas, i, j)
        coeff = kij * charges[i] * charges[j] / c^2 * ṙ / r^2
        p[i] .-= coeff .* Δr
    end
    return p
end

"""
    verify_ic(positions, velocities, masses, charges, kappas, c)

Runs the §7 6-step checklist. Asserts the exact (F) consistency identity
holds to 1e-10. Returns `(E, T0, U0, H_Wesley, p_initial)`.
"""
function verify_ic(positions, velocities, masses, charges, kappas, c)
    n = length(positions)
    M_tot = sum(masses)

    R_com = sum(masses[i] .* positions[i]  for i in 1:n) ./ M_tot
    P_com = sum(masses[i] .* velocities[i] for i in 1:n)
    println("Step 1 — COM and total physical momentum")
    println("  Σ m_i r_i / M = ", round.(R_com, digits=10))
    println("  Σ m_i v_i     = ", round.(P_com, digits=10))

    rij  = Dict{Tuple{Int,Int},Float64}()
    rdot = Dict{Tuple{Int,Int},Float64}()
    println("\nStep 2 — pair radial rates ṙ_ij")
    for i in 1:n-1, j in i+1:n
        Δr = positions[i] .- positions[j]
        r  = norm(Δr)
        Δv = velocities[i] .- velocities[j]
        ṙ  = dot(Δr, Δv) / r
        rij[(i,j)] = r; rdot[(i,j)] = ṙ
        println(@sprintf("  (%d,%d): r = %.6f, ṙ = %+.6e", i, j, r, ṙ))
    end

    T_phys = 0.5 * sum(masses[i] * dot(velocities[i], velocities[i]) for i in 1:n)
    U_Weber = 0.0
    for i in 1:n-1, j in i+1:n
        kij = _kappa(kappas, i, j)
        U_Weber += kij * charges[i] * charges[j] / rij[(i,j)] *
                   (1 - rdot[(i,j)]^2 / (2*c^2))
    end
    E = T_phys + U_Weber
    println("\nStep 3 — physical Hamiltonian")
    println(@sprintf("  T_phys  = %+.6f", T_phys))
    println(@sprintf("  U_Weber = %+.6f", U_Weber))
    println(@sprintf("  E       = %+.6f   (bound? %s)", E, E < 0 ? "✓" : "✗"))

    p_can = canonical_momentum(positions, velocities, masses, charges, kappas, c)
    P_can = sum(p_can)
    println("\nStep 4 — canonical momenta and Σ p_i")
    for i in 1:n
        println(@sprintf("  p_%d = %s", i, round.(p_can[i], digits=10)))
    end
    println("  Σ p_i = ", round.(P_can, digits=10))

    sum_psq = sum(dot(p_can[i], p_can[i]) / (2*masses[i]) for i in 1:n)
    Δ_meas  = sum_psq - T_phys
    Δ_pred  = 0.0
    for i in 1:n-1, j in i+1:n
        kij = _kappa(kappas, i, j)
        Δ_pred -= kij * charges[i] * charges[j] / c^2 * rdot[(i,j)]^2 / rij[(i,j)]
    end
    for i in 1:n
        X_i = zero(positions[i])
        for j in 1:n
            i == j && continue
            Δr  = positions[i] .- positions[j]
            kij = _kappa(kappas, i, j)
            key = (min(i,j), max(i,j))
            coeff = kij * charges[i] * charges[j] / c^2 * rdot[key] / rij[key]^2
            X_i .+= coeff .* Δr
        end
        Δ_pred += dot(X_i, X_i) / (2*masses[i])
    end
    err_5 = abs(Δ_meas - Δ_pred)
    println("\nStep 5 — exact (F) identity")
    println(@sprintf("  Δ_meas=%+.6e, Δ_pred=%+.6e, |Δ_meas−Δ_pred|=%.3e  %s",
                     Δ_meas, Δ_pred, err_5, err_5 < 1e-10 ? "✓" : "✗"))
    @assert err_5 < 1e-10 "(F) identity violated"

    rdot_p = Dict{Tuple{Int,Int},Float64}()
    for i in 1:n-1, j in i+1:n
        Δr   = positions[i] .- positions[j]
        v_pi = p_can[i] ./ masses[i]
        v_pj = p_can[j] ./ masses[j]
        rdot_p[(i,j)] = dot(Δr, v_pi .- v_pj) / rij[(i,j)]
    end
    U_W_p = 0.0
    for i in 1:n-1, j in i+1:n
        kij = _kappa(kappas, i, j)
        U_W_p += kij * charges[i] * charges[j] / rij[(i,j)] *
                 (1 - rdot_p[(i,j)]^2 / (2*c^2))
    end
    H_Wesley = sum_psq + U_W_p
    println("\nStep 6 — Wesley H (integrator-conserved) vs E_phys")
    println(@sprintf("  E_phys   = %+.10e", E))
    println(@sprintf("  H_Wesley = %+.10e   (Δ = %.3e, %.2f%% of |E|)",
                     H_Wesley, H_Wesley - E, 100 * abs(H_Wesley - E) / abs(E)))
    return (E=E, T0=T_phys, U0=U_Weber, H_Wesley=H_Wesley, p_initial=p_can)
end

flatten_state(vectors_per_particle) = vcat(vectors_per_particle...)

flatten_state (generic function with 1 method)

## 2. System parameters

Asymmetric charges ($q_1=+1$, $q_2=-2$, $q_3=+1$) keep the inner binary
bound ($k_{12}=-2$) and give the orbiter a nonzero outer coupling
$k_\mathrm{out}=-1$ without needing Zöllner. Inner binary is at
eccentric mid-flight ($e_B=0.5$, $s_B=+1$ outbound); outer orbit is
Method-A circular at radius $R=20$.

In [4]:
# Charges, masses, c
m1_3, m2_3, m3_3 = 1.0, 1.0, 1.0
q1_3, q2_3, q3_3 = +1.0, -2.0, +1.0
c_3 = 4.0

mu12   = m1_3 * m2_3 / (m1_3 + m2_3)
k12    = q1_3 * q2_3                      # -2, attractive
M_in   = m1_3 + m2_3
M_to   = m1_3 + m2_3 + m3_3
mu_out = (m1_3 + m2_3) * m3_3 / M_to
k_out  = (q1_3 + q2_3) * q3_3             # -1, attractive

# Inner binary: e=0.5, E_B = -0.3·|k₁₂|/r₀, mid-flight outbound
r0   = 1.0
E_B  = -0.3 * abs(k12) / r0
e_B  =  0.5
L_B  = sqrt(mu12 * k12^2 * (1 - e_B^2) / (2 * abs(E_B)))
a_B  = k12 / (2*E_B)
T_B  = kepler_period(a_B, mu12, k12)
(rp_B, ra_B) = kepler_apsides(E_B, L_B, k12, mu12)
@assert rp_B <= r0 <= ra_B "r0 out of binary apsides"

# Outer orbit: Method-A circular at radius R_o
R_o   = 20.0
V_rel = sqrt(abs(k_out) / (mu_out * R_o))
T_out = 2π * R_o / V_rel

@printf("Inner: μ=%.4f, k=%g, E_B=%g, L_B=%.4f, e=%.2f, a=%.4f, T_B=%.4f\n",
        mu12, k12, E_B, L_B, e_B, a_B, T_B)
@printf("       r_p=%.4f, r_a=%.4f, r_0=%.4f\n", rp_B, ra_B, r0)
@printf("Outer: μ=%.4f, k=%g, R=%g, V_rel=%.4f, T_out=%.4f\n",
        mu_out, k_out, R_o, V_rel, T_out)
@printf("Hierarchy: T_out / T_B = %.1f, r0/R = %.4f\n", T_out / T_B, r0 / R_o)

Inner: μ=0.5000, k=-2, E_B=-0.6, L_B=1.1180, e=0.50, a=1.6667, T_B=6.7596
       r_p=0.8333, r_a=2.5000, r_0=1.0000
Outer: μ=0.6667, k=-1, R=20, V_rel=0.2739, T_out=458.8590
Hierarchy: T_out / T_B = 67.9, r0/R = 0.0500


## 3. Build the IC (§2.4 forward map applied)

Binary separation along $\hat z$, tangential velocity (from $L_B$) along
$\hat x$. Binary COM sits at $-\,(m_3/M_\mathrm{tot})R_o\,\hat x$ and
orbiter at $+\,(M_\mathrm{in}/M_\mathrm{tot})R_o\,\hat x$, so the system
COM is at the origin. Outer circular velocities along $\hat y$, opposite
for the binary COM and the orbiter so $\sum \vec p = \vec 0$.

In [5]:
# Inner binary: mid-flight outbound
s_B = +1
ṙ_B = radial_velocity(E_B, L_B, r0, k12, mu12, c_3, s_B)
v⊥_B = L_B / (mu12 * r0)

# v_rel = ṙ ẑ + v_⊥ x̂
v_rel_B = [v⊥_B, 0.0, ṙ_B]
v1_int  = +(m2_3 / M_in) .* v_rel_B
v2_int  = -(m1_3 / M_in) .* v_rel_B

# Positions
R_B   = [-(m3_3/M_to) * R_o, 0.0, 0.0]    # binary COM
r1_3  = R_B .+ [0.0, 0.0, -(m2_3/M_in) * r0]
r2_3  = R_B .+ [0.0, 0.0, +(m1_3/M_in) * r0]
r3_3  = [+(M_in/M_to) * R_o, 0.0, 0.0]

# Outer circulation: V_B along +ŷ, v3 along -ŷ (zero total momentum)
V_B  = [0.0, +(m3_3/M_to) * V_rel, 0.0]
v3_3 = [0.0, -(M_in/M_to) * V_rel, 0.0]
v1_3 = v1_int .+ V_B
v2_3 = v2_int .+ V_B

@printf("Inner binary: ṙ_B = %+.6f, v_⊥,B = %+.6f\n", ṙ_B, v⊥_B)

Inner binary: ṙ_B = +0.516398, v_⊥,B = +2.236068


In [6]:
diag_3 = verify_ic([r1_3, r2_3, r3_3], [v1_3, v2_3, v3_3],
                   [m1_3, m2_3, m3_3], [q1_3, q2_3, q3_3], 1.0, c_3)
nothing

Step 1 — COM and total physical momentum
  Σ m_i r_i / M = [0.0, 0.0, 0.0]
  Σ m_i v_i     = [0.0, 0.0, 0.0]

Step 2 — pair radial rates ṙ_ij
  (1,2): r = 1.000000, ṙ = -5.163978e-01
  (1,3): r = 20.006249, ṙ = -1.124138e+00
  (2,3): r = 20.006249, ṙ = +1.111232e+00

Step 3 — physical Hamiltonian
  T_phys  = +1.341667
  U_Weber = -2.031434
  E       = -0.689767   (bound? ✓)

Step 4 — canonical momenta and Σ p_i
  p_1 = [1.1145232526, 0.0912870929, 0.3226608438]
  p_2 = [-1.1249748495, 0.0912870929, -0.3225750907]
  p_3 = [0.010451597, -0.1825741858, -8.57531e-5]
  Σ p_i = [-0.0, 0.0, -0.0]

Step 5 — exact (F) identity
  Δ_meas=+4.133556e-02, Δ_pred=+4.133556e-02, |Δ_meas−Δ_pred|=2.776e-17  ✓

Step 6 — Wesley H (integrator-conserved) vs E_phys
  E_phys   = -6.8976728342e-01
  H_Wesley = -6.3892435047e-01   (Δ = 5.084e-02, 7.37% of |E|)


## 4. Build the `HamiltonianProblem` (streaming-ready)

`tspan=(0.0, Inf)` enables the streaming animator's internal recycling
loop. Pick `dt=5e-3` to match the parent notebook's Scenario 3 (small
enough that binary oscillation is resolved; large enough to animate at a
reasonable rate).

Flip `run_animation = false` if you only want to `nbconvert --execute`
this notebook without opening a GLMakie window (the live cell would
otherwise block indefinitely).

In [7]:
sys_3   = HamiltonianSystem(3, 3)
q0_3    = flatten_state([r1_3, r2_3, r3_3])
p0_3    = flatten_state(diag_3.p_initial)
dt_3    = 5e-3
prob_3  = HamiltonianProblem(sys_3, (0.0, Inf), q0_3, p0_3;
    masses=[m1_3, m2_3, m3_3], charges=[q1_3, q2_3, q3_3], c=c_3, dt=dt_3)

run_animation = true   # set to false for headless nbconvert
println("Problem ready: 3 particles, 3D, dt=", dt_3)

Problem ready: 3 particles, 3D, dt=0.005



## 5. Live 3D animation

`tail_length=400` keeps enough history to see the binary's ẑ trace while
the orbiter takes a significant slice of its $T_\mathrm{out}$ sweep.
Drag on the `Axis3` with the mouse to rotate; the **Speed** slider
controls integration steps per frame; the **Phase** dropdown switches
the sidebar view between pair-separation portraits and per-particle
phase-space coordinates.

In [8]:
if run_animation
    animate_weber(prob_3;
        buffer_size = 4000,
        tail_length = 400,
        compute_batch = 2)
else
    @info "run_animation=false; skipping live animation."
end

GLMakie.Screen(...)